In [ ]:
import pandas as pd
from pathlib import Path
from scipy import stats

# where you saved the files
CORR_FILE   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\df_sameW.csv")  #IMPORTANT: I should actually use the one called df_sameW_goodlag
BEH_DIR    = Path(r"C:\Users\cdd\Documents\Uni\Special_course\ds003838-download")

SUBJECTS   = ["sub-034"]      # put all subjects here when you’re ready


In [18]:
corr = (pd.read_csv(CORR_FILE)
          .assign(                       # extract trial #, then +1 to match *.tsv
              trial=lambda d:
                  d.epoch.str.extract(r"trial_(\d+)\.csv")
                    .astype(int).squeeze()
                    + 1                 # 0-based → 1-based
          ))


In [19]:
def explode_triggers(df_beh):
    out_frames = []
    for _, row in df_beh.iterrows():
        trig = row["triggerCorrect"].strip()
        out_frames.append(pd.DataFrame({
            "trial"     : row["trial"],
            "digit_pos" : range(1, len(trig) + 1),   # 1, 2, …
            "recalled"  : [int(c) for c in trig]      # 0 / 1
        }))
    return pd.concat(out_frames, ignore_index=True)


In [20]:
subject_dfs = []      # collect for an optional grand-average
stats_per_sub = []    # store per-subject t-tests

for sub in corr["subject"].unique():
    # --- Behaviour ------------------------------------------------------
    beh_path = (BEH_DIR / sub / "beh" / f"{sub}_task-memory_beh.tsv")
    beh_raw  = pd.read_csv(beh_path, sep="\t", usecols=["trial", "triggerCorrect"], dtype = {"triggerCorrect": str})
    beh      = explode_triggers(beh_raw)

    # --- Merge ----------------------------------------------------------
    merged = corr.query("subject == @sub").merge(
                 beh, on=["trial", "digit_pos"], how="inner")

    # --- Stats ----------------------------------------------------------
    rec  = merged.loc[merged.recalled == 1, "r"]
    miss = merged.loc[merged.recalled == 0, "r"]

    t, p = stats.ttest_ind(rec, miss, equal_var=False)
    stats_per_sub.append(
        {"subject": sub,
         "mean_r_recalled": rec.mean(),
         "mean_r_missed"  : miss.mean(),
         "t" : t, "p" : p, "n_recalled": len(rec), "n_missed": len(miss)}
    )

    merged["subject"] = sub
    subject_dfs.append(merged)


In [21]:
pd.set_option("display.precision", 3)
print(pd.DataFrame(stats_per_sub))

    subject  mean_r_recalled  mean_r_missed      t      p  n_recalled  \
0   sub-033       -9.420e-04      3.238e-02 -1.732  0.084         357   
1   sub-034        4.122e-02      7.105e-02    NaN    NaN         567   
2   sub-035        1.734e-01      7.864e-02  1.265  0.212          75   
3   sub-036        3.791e-03      5.564e-02 -1.320  0.189          89   
4   sub-038        3.020e-02      5.988e-02    NaN    NaN         159   
5   sub-039        4.628e-02      2.475e-02  0.838  0.402         360   
6   sub-040       -3.901e-03      1.874e-02 -0.872  0.384         385   
7   sub-041        1.891e-02     -1.497e-03  1.026  0.305         448   
8   sub-042        2.802e-02      2.960e-03  1.184  0.237         532   
9   sub-043        1.336e-02      2.095e-02    NaN    NaN         363   
10  sub-044        2.758e-02      2.209e-02  0.281  0.779         426   
11  sub-045        3.490e-02      6.662e-02 -1.292  0.197         203   
12  sub-046        1.602e-01      9.571e-02    NaN 

In [22]:
print(f"sub {sub}:  n_recalled = {len(rec)},  var = {rec.var(ddof=1):.3g}")
print(f"sub {sub}:  n_missed   = {len(miss)}, var = {miss.var(ddof=1):.3g}")


sub sub-098:  n_recalled = 173,  var = 0.0703
sub sub-098:  n_missed   = 214, var = 0.0715


In [23]:
import statsmodels.formula.api as smf
md = smf.mixedlm("r ~ recalled", group_df, groups=group_df["subject"])
print(md.fit().summary())

          Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: r          
No. Observations: 39581   Method:             REML       
No. Groups:       57      Scale:              0.1059     
Min. group size:  5       Log-Likelihood:     -11791.5542
Max. group size:  972     Converged:          Yes        
Mean group size:  694.4                                  
----------------------------------------------------------
            Coef.  Std.Err.    z     P>|z|  [0.025  0.975]
----------------------------------------------------------
Intercept   0.053     0.005  10.185  0.000   0.043   0.063
recalled    0.004     0.003   1.121  0.262  -0.003   0.010
Group Var   0.001     0.001                               



c:\Users\cdd\AppData\Local\anaconda3\envs\special_course_env\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [26]:
# table of means per subject × recalled(0/1)
group_df = pd.concat(subject_dfs, ignore_index=True)
sub_means = (group_df.groupby(["subject", "recalled"])["r"].mean().unstack())        # columns 0 = missed, 1 = recalled

# drop any subject lacking one category (all-correct or all-wrong)
sub_means = sub_means.dropna(subset=[0, 1])

# Calculate differences
diffs = sub_means[1] - sub_means[0]
mean_diff = diffs.mean()
std_diff = diffs.std(ddof=1)
n = len(diffs)
se_diff = std_diff / (n ** 0.5)  # Standard error

# paired t-test (recalled − missed)
from scipy.stats import ttest_rel, t
t_stat, p_two = ttest_rel(sub_means[1], sub_means[0])
p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2

# Calculate 10% CI
alpha = 0.05
df = n - 1  # Degrees of freedom
t_critical = t.ppf(1 - alpha/2, df)
ci_lower = mean_diff - t_critical * se_diff
ci_upper = mean_diff + t_critical * se_diff

print(f"Paired: t = {t_stat:.3f}, one-tailed p = {p_one:.4g}, "
      f"mean Δr = {mean_diff:.3f}  (N_subjects = {n})")
print(f"95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")

# Cohen's d for paired samples
cohens_d = mean_diff / std_diff  # std_diff is already the SD of the differences

# Alternatively, directly from t-statistic:
cohens_d_from_t = t_stat / (n ** 0.5)

print(f"Cohen's d (paired) = {cohens_d:.3f}")
print(f"Cohen's d (from t) = {cohens_d_from_t:.3f}")



Paired: t = 1.754, one-tailed p = 0.04249, mean Δr = 0.007  (N_subjects = 56)
95% CI: [-0.001, 0.016]
Cohen's d (paired) = 0.234
Cohen's d (from t) = 0.234


In [25]:
# ------------------------------------------------------------------
# How many recalled / missed digits does each subject contribute?
# ------------------------------------------------------------------
digit_counts = (group_df
                .groupby(["subject", "recalled"])
                .size()                      # count rows
                .unstack(fill_value=0)       # columns 0 = missed, 1 = recalled
                .rename(columns={0: "missed", 1: "recalled"}))

median_rec   = digit_counts["recalled"].mean()
median_miss  = digit_counts["missed"].mean()

print(f"Median per subject: {median_rec:.0f} recalled, {median_miss:.0f} missed")


Median per subject: 316 recalled, 379 missed
